# P5 — Elexon Generation Data: Full Extraction & Spatial Split

Locked decisions this notebook builds against (not re-litigated here):

- **Sources**: Elexon Insights Solution API only (public, no key, Stream endpoints) — never the legacy key-gated BMRS API or ElexonDataPortal package.
- **Fuel-type scope**: mixed GB fleet, all fuel types. This is a weaker geographic story than a single fuel type would give — that tension is acknowledged here and must be stated explicitly in the eventual report, not glossed over.
- **Spatial unit / split key**: location (`dictionary_id`), not individual BMU. Multi-BMU locations (e.g. Didcot) share one set of coordinates and are never split across train/test — every fuel-series belonging to one location moves together as an atomic block.
- **Resolution/span**: half-hourly, full available history. The exploration notebook found an apparent ~2019-02-01 backfill floor on a 6-site sample — this notebook confirms or corrects that at full-fleet scale rather than assuming it generalises.
- **Raw storage**: one CSV per BMU under `data/raw/generation/`, source of truth. A consolidated parquet file is a later, disposable, regenerable convenience cache for the training notebook — never the source of truth.
- **Empty/mothballed BMUs**: detected and logged generically via the manifest (zero rows returned), not hand-excluded by name.

**Revision log (2026-08-06, on the back of the first full run's results):**
1. Aggregation unit changed from `dictionary_id` alone to **`(dictionary_id, fuel_type)`** — individual fuel signals are preserved rather than blended into a `MIXED` series, since the data-scarcity concern outweighs the cleaner single-series-per-location story. See Step 3.
2. `dictionary_id` **remains the split key** regardless — a location's fuel-series are never split across train/test.
3. Convex hull now computed once per unique location (213 of them), then joined onto every fuel-series at that location — not recomputed per fuel-series. See Step 4.
4. Stratified holdout fuel types confirmed at ≥7 interior sites: **WIND, CCGT, NPSHYD, NUCLEAR** — re-checked against the revised Step 5 table in case the new granularity pushes any other fuel type over that line.
5. A minimum-history cutoff will be applied, but the threshold itself isn't chosen here — Step 6 proposes candidates with the cost of each, for Simon to pick from.

This notebook stops short of: picking the minimum-history cutoff, or treating the parquet cache as anything other than derived. Those are for Simon to confirm once the revised reports below are available.

In [1]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO
from datetime import date, datetime, timedelta
import time
import json

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RAW_DIR = Path("../data/raw/generation")
RAW_DIR.mkdir(parents=True, exist_ok=True)
REF_DIR = Path("../data/reference")
REF_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

OSUKED_RAW = "https://raw.githubusercontent.com/OSUKED/Power-Station-Dictionary/main"
ELEXON_API = "https://data.elexon.co.uk/bmrs/api/v1"

REQUEST_DELAY_S = 0.1  # be a good netizen
MANIFEST_PATH = INTERIM_DIR / "extraction_manifest.csv"

# Global chunk floor: deliberately earlier than the ~2019-02-01 floor the
# 6-site exploration sample suggested, so the full pull can confirm or
# correct that rather than assume it. See Step 6 for the confirmed figure.
EXTRACTION_START_YEAR = 2015
EXTRACTION_END = date.today() - timedelta(days=10)  # ~5 working day publication lag

## Step 1 — Resumable extraction pipeline design

**Manifest** (`data/interim/extraction_manifest.csv`): one row per `(bmu, chunk_start, chunk_end)` attempted. Columns: `ngc_bmu_id`, `chunk_start`, `chunk_end`, `status` (`pending`/`success`/`failed`), `rows_returned`, `attempted_at`, `error`.

**Chunking**: one calendar year per chunk, per BMU, from `EXTRACTION_START_YEAR` to the present (minus the publication lag). This keeps individual requests small and makes retries cheap.

**Resume logic**: the manifest is loaded (not rebuilt) on every run. Rows already `success` are skipped. Rows `failed` or `pending` are retried/attempted. This means an interrupted run can simply be re-run.

**Raw storage**: each successful chunk's rows are appended straight to `data/raw/generation/{ngc_bmu_id}.csv` as soon as they're fetched — so data already pulled is durable on disk even if the run is interrupted mid-way, not just held in memory until the end.

**Retries**: HTTP 429/5xx are treated as transient — retried with a short backoff (up to 3 attempts) rather than marked `failed` immediately. Other errors (4xx other than 429) are logged as `failed` and not retried automatically.

**Politeness**: a fixed 0.1s delay between requests.

In [2]:
# Fresh pull of the OSUKED reference tables (the complete BMU universe for this
# extraction comes from fuel_types, per the brief - not just the exploration sample)
osuked_files = {
    "fuel_types": "data/linked-datapackages/bmu-fuel-types/fuel_types.csv",
    "plant_locations": "data/linked-datapackages/plant-locations/plant-locations.csv",
    "dictionary_ids": "data/dictionary/ids.csv",
}

osuked = {}
for name, path in osuked_files.items():
    resp = requests.get(f"{OSUKED_RAW}/{path}", timeout=30)
    resp.raise_for_status()
    osuked[name] = pd.read_csv(StringIO(resp.text))

bmu_universe = sorted(osuked["fuel_types"]["ngc_bmu_id"].dropna().str.strip().unique())
print(f"Full BMU universe from OSUKED fuel_types: {len(bmu_universe)} distinct BMUs")

Full BMU universe from OSUKED fuel_types: 462 distinct BMUs


In [3]:
MANIFEST_COLUMNS = ["ngc_bmu_id", "chunk_start", "chunk_end", "status", "rows_returned", "attempted_at", "error"]


def year_chunks(start_year: int, end_date: date):
    """Yield (chunk_start, chunk_end) date strings, one per calendar year."""
    chunks = []
    for year in range(start_year, end_date.year + 1):
        chunk_start = date(year, 1, 1)
        chunk_end = date(year, 12, 31) if year < end_date.year else end_date
        if chunk_start > end_date:
            break
        chunks.append((chunk_start.isoformat(), chunk_end.isoformat()))
    return chunks


def load_or_init_manifest(bmus, start_year=EXTRACTION_START_YEAR, end_date=EXTRACTION_END):
    if MANIFEST_PATH.exists():
        manifest = pd.read_csv(MANIFEST_PATH, dtype={"ngc_bmu_id": str})
    else:
        manifest = pd.DataFrame(columns=MANIFEST_COLUMNS)

    existing_keys = set(zip(manifest["ngc_bmu_id"], manifest["chunk_start"], manifest["chunk_end"]))
    new_rows = []
    for bmu in bmus:
        for chunk_start, chunk_end in year_chunks(start_year, end_date):
            key = (bmu, chunk_start, chunk_end)
            if key not in existing_keys:
                new_rows.append({
                    "ngc_bmu_id": bmu, "chunk_start": chunk_start, "chunk_end": chunk_end,
                    "status": "pending", "rows_returned": pd.NA, "attempted_at": pd.NA, "error": pd.NA,
                })
    if new_rows:
        manifest = pd.concat([manifest, pd.DataFrame(new_rows)], ignore_index=True)
        manifest.to_csv(MANIFEST_PATH, index=False)
    return manifest


def save_manifest(manifest):
    manifest.to_csv(MANIFEST_PATH, index=False)

In [4]:
RETRYABLE_STATUS = {429, 500, 502, 503, 504}


def fetch_chunk(bmu, chunk_start, chunk_end, max_attempts=3):
    """Fetch one (bmu, year) chunk from the B1610 stream endpoint.
    Retries 429/5xx with backoff; other HTTP errors (e.g. 400/404) fail immediately.
    """
    params = {"from": chunk_start, "to": chunk_end, "bmUnit": bmu}
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            r = requests.get(f"{ELEXON_API}/datasets/B1610/stream", params=params, timeout=60)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            last_exc = exc
            if attempt < max_attempts:
                time.sleep(2 ** attempt)  # 2s, 4s backoff
            continue

        if r.status_code in RETRYABLE_STATUS:
            last_exc = requests.exceptions.HTTPError(f"retryable status {r.status_code}")
            if attempt < max_attempts:
                time.sleep(2 ** attempt)
            continue

        r.raise_for_status()  # non-retryable 4xx raises immediately, no retry loop
        return r.json()
    raise last_exc


def append_bmu_csv(bmu, records):
    if not records:
        return
    df = pd.DataFrame(records)
    path = RAW_DIR / f"{bmu}.csv"
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)

In [5]:
def run_extraction(manifest, progress_every=25):
    """Work through every pending/failed row in the manifest, in place.
    Prints a progress line with a rough ETA every `progress_every` chunks attempted.
    Safe to interrupt and re-run: already-`success` rows are untouched here.
    """
    todo_mask = manifest["status"].isin(["pending", "failed"])
    todo_idx = manifest.index[todo_mask].tolist()
    total_todo = len(todo_idx)
    start_time = time.monotonic()
    total_rows_this_run = 0

    print(f"{total_todo} chunks to attempt ({(~todo_mask).sum()} already succeeded previously)")

    for n, idx in enumerate(todo_idx, start=1):
        row = manifest.loc[idx]
        bmu, chunk_start, chunk_end = row["ngc_bmu_id"], row["chunk_start"], row["chunk_end"]
        try:
            records = fetch_chunk(bmu, chunk_start, chunk_end)
            append_bmu_csv(bmu, records)
            manifest.loc[idx, ["status", "rows_returned", "attempted_at", "error"]] = (
                "success", len(records), datetime.now().isoformat(timespec="seconds"), pd.NA,
            )
            total_rows_this_run += len(records)
        except Exception as exc:
            manifest.loc[idx, ["status", "rows_returned", "attempted_at", "error"]] = (
                "failed", pd.NA, datetime.now().isoformat(timespec="seconds"), str(exc)[:200],
            )
        time.sleep(REQUEST_DELAY_S)

        if n % progress_every == 0 or n == total_todo:
            save_manifest(manifest)  # flush periodically so progress is durable, not just in memory
            elapsed = time.monotonic() - start_time
            rate = n / elapsed  # chunks/sec
            remaining = total_todo - n
            eta_s = remaining / rate if rate > 0 else float("nan")
            eta_min = eta_s / 60
            done_bmus = manifest.loc[manifest["status"] == "success", "ngc_bmu_id"].nunique()
            print(
                f"[{n}/{total_todo}] {n/total_todo:.1%} | "
                f"{done_bmus}/{len(bmu_universe)} BMUs with >=1 success | "
                f"{total_rows_this_run:,} rows this run | "
                f"{rate:.2f} chunks/s | ETA ~{eta_min:.1f} min"
            )

    save_manifest(manifest)
    return manifest

## Step 2 — Kick off the full pull

Across the complete BMU universe (all `ngc_bmu_id` values in OSUKED `fuel_types`), not just the 6-BMU exploration sample. This is expected to run long (hundreds of BMUs × up to 12 yearly chunks each) — that's fine, the manifest makes it resumable, and the progress line every 25 chunks includes a rate-based ETA so it's easy to tell it's making steady progress rather than stuck.

In [6]:
manifest = load_or_init_manifest(bmu_universe)
print(f"Manifest: {len(manifest)} total chunks, {(manifest['status'] == 'success').sum()} already successful")

manifest = run_extraction(manifest)

print()
print("Pull finished (or manifest fully attempted this run).")
print(manifest["status"].value_counts())

Manifest: 5544 total chunks, 5544 already successful
0 chunks to attempt (5544 already succeeded previously)

Pull finished (or manifest fully attempted this run).
status
success    5544
Name: count, dtype: int64


## Step 3 — Aggregation by (location, fuel type)

Rebuilding the confirmed join path from the exploration notebook: `plant_locations.dictionary_id -> dictionary_ids.dictionary_id -> dictionary_ids.ngc_bmu_id (exploded) -> fuel_types.ngc_bmu_id`. This gives every BMU's owning location and coordinates.

**Revised aggregation unit: `(dictionary_id, fuel_type)`, not `dictionary_id` alone.** BMUs are still summed (`quantity`, in MWh) across `settlementDate` + `settlementPeriod`, but only within the same fuel type at a location — a multi-technology location like Didcot now produces a separate series per fuel type (CCGT, OCGT) rather than one blended series. Individual fuel signals are preserved, not blended: the data-scarcity concern outweighs the cleaner single-series-per-location story.

`dictionary_id` remains the **split key** — every fuel-series belonging to one location is assigned to train or test as a single atomic block; a location is never split across train and test just because it has more than one fuel-series. `site_fuel_id` (`{dictionary_id}_{fuel_type}`) identifies the individual series; `dictionary_id` identifies the location.

In [7]:
# Rebuild the join path from exploration
ids = osuked["dictionary_ids"][["dictionary_id", "name", "ngc_bmu_id"]].copy()
ids = ids.dropna(subset=["ngc_bmu_id"])
ids["ngc_bmu_id"] = ids["ngc_bmu_id"].str.split(",")
ids_exploded = ids.explode("ngc_bmu_id")
ids_exploded["ngc_bmu_id"] = ids_exploded["ngc_bmu_id"].str.strip()

site_bmu_map = (
    osuked["plant_locations"]
    .merge(ids_exploded, on="dictionary_id", how="inner")
    .merge(osuked["fuel_types"], on="ngc_bmu_id", how="inner")
)
# Only keep BMUs we actually attempted to pull data for
site_bmu_map = site_bmu_map[site_bmu_map["ngc_bmu_id"].isin(bmu_universe)]

print(f"site_bmu_map: {len(site_bmu_map)} BMU rows across {site_bmu_map['dictionary_id'].nunique()} sites")
site_bmu_map.head()

site_bmu_map: 403 BMU rows across 213 sites


,dictionary_id,longitude,latitude,name,ngc_bmu_id,fuel_type,comments
0,10000,-3.603516,57.480403,Rothes Bio-Plant CHP,MARK-1,BIOMASS,NaN
1,10000,-3.603516,57.480403,Rothes Bio-Plant CHP,MARK-2,BIOMASS,NaN
2,10001,-1.267570,51.623630,Didcot,DIDC01G,OCGT,NaN
3,10001,-1.267570,51.623630,Didcot,DIDC02G,OCGT,NaN
4,10001,-1.267570,51.623630,Didcot,DIDC03G,OCGT,NaN


In [8]:
# Under the revised (dictionary_id, fuel_type) aggregation, a location with more
# than one fuel type is no longer a problem to flag - it just becomes more than
# one fuel-series. Reported here for context, not as a data-quality concern.
fuel_types_per_location = site_bmu_map.groupby("dictionary_id")["fuel_type"].nunique()
multi_fuel_locations = fuel_types_per_location[fuel_types_per_location > 1]

print(f"{len(multi_fuel_locations)} location(s) have BMUs spanning more than one fuel type - "
      f"each now becomes its own fuel-series rather than being blended:")
display(
    site_bmu_map[site_bmu_map["dictionary_id"].isin(multi_fuel_locations.index)]
    .sort_values(["dictionary_id", "fuel_type"])[["dictionary_id", "name", "fuel_type"]]
    .drop_duplicates(["dictionary_id", "fuel_type"])
)

13 location(s) have BMUs spanning more than one fuel type - each now becomes its own fuel-series rather than being blended:


,dictionary_id,name,fuel_type
6,10001,Didcot,CCGT
2,10001,Didcot,OCGT
8,10002,Aberthaw B,COAL
11,10002,Aberthaw B,OCGT
18,10004,Drax,BIOMASS
22,10004,Drax,COAL
24,10004,Drax,OCGT
31,10006,Ferrybridge C,COAL
35,10006,Ferrybridge C,OCGT
37,10007,Fiddlers Ferry,COAL


In [9]:
SITE_GEN_DIR = INTERIM_DIR / "site_generation"
SITE_GEN_DIR.mkdir(parents=True, exist_ok=True)


def aggregate_site_fuel(dictionary_id, fuel_type, bmus, longitude, latitude):
    """Sum half-hourly generation across the BMUs sharing one (dictionary_id, fuel_type).
    Returns None if none of these BMUs have any raw data pulled yet."""
    frames = []
    for bmu in bmus:
        path = RAW_DIR / f"{bmu}.csv"
        if path.exists():
            df = pd.read_csv(path, usecols=["settlementDate", "settlementPeriod", "quantity"])
            frames.append(df)
    if not frames:
        return None
    combined = pd.concat(frames, ignore_index=True)
    # Guard against any duplicate rows from a chunk retried after a partial write
    combined = combined.drop_duplicates()
    series = (
        combined.groupby(["settlementDate", "settlementPeriod"], as_index=False)["quantity"]
        .sum()
        .assign(dictionary_id=dictionary_id, fuel_type=fuel_type, longitude=longitude, latitude=latitude)
    )
    return series


def build_all_site_fuel_series(site_bmu_map, save=True):
    summaries = []
    for (dictionary_id, fuel_type), group in site_bmu_map.groupby(["dictionary_id", "fuel_type"]):
        longitude, latitude = group["longitude"].iloc[0], group["latitude"].iloc[0]
        series = aggregate_site_fuel(dictionary_id, fuel_type, group["ngc_bmu_id"].tolist(), longitude, latitude)
        if series is None:
            continue
        site_fuel_id = f"{dictionary_id}_{fuel_type}"
        if save:
            series.to_csv(SITE_GEN_DIR / f"{site_fuel_id}.csv", index=False)
        summaries.append({
            "site_fuel_id": site_fuel_id,
            "dictionary_id": dictionary_id,  # location / split key - never split within this
            "name": group["name"].iloc[0],
            "longitude": longitude,
            "latitude": latitude,
            "fuel_type": fuel_type,
            "n_bmus": group["ngc_bmu_id"].nunique(),
            "n_half_hours": len(series),
            "earliest_settlement_date": series["settlementDate"].min(),
            "latest_settlement_date": series["settlementDate"].max(),
        })
    return pd.DataFrame(summaries)


site_fuel_summary = build_all_site_fuel_series(site_bmu_map)
n_possible = site_bmu_map.groupby(["dictionary_id", "fuel_type"]).ngroups
print(f"Fuel-series with at least some pulled data: {len(site_fuel_summary)} / {n_possible} possible (location, fuel type) combinations")
print(f"Distinct locations represented: {site_fuel_summary['dictionary_id'].nunique()}")
site_fuel_summary.head()

Fuel-series with at least some pulled data: 199 / 228 possible (location, fuel type) combinations
Distinct locations represented: 189


,site_fuel_id,dictionary_id,name,longitude,latitude,fuel_type,n_bmus,n_half_hours,earliest_settlement_date,latest_settlement_date
0,10000_BIOMASS,10000,Rothes Bio-Plant CHP,-3.603516,57.480403,BIOMASS,2,130808,2019-02-01,2026-07-27
1,10001_CCGT,10001,Didcot,-1.267570,51.623630,CCGT,2,130808,2019-02-01,2026-07-27
2,10001_OCGT,10001,Didcot,-1.267570,51.623630,OCGT,4,130808,2019-02-01,2026-07-27
3,10004_BIOMASS,10004,Drax,-0.996631,53.736634,BIOMASS,4,130808,2019-02-01,2026-07-27
4,10004_COAL,10004,Drax,-0.996631,53.736634,COAL,2,130808,2019-02-01,2026-07-27


**Observations:** 199 fuel-series have data, out of 228 possible `(location, fuel type)` combinations — the other 29 are zero-data (mothballed BMUs, same pattern as before, just now counted per fuel-series rather than per location). Those 199 series span 189 distinct locations, matching the previous location-with-data count exactly — so splitting multi-fuel locations into separate series didn't change *which* locations have data, only how many series each contributes.

13 locations were confirmed to have BMUs spanning more than one fuel type (same 13 as before — Didcot, Drax, Fiddlers Ferry, Littlebrook D, etc.) — these now contribute one series per fuel type rather than one blended series. No longer a concern to flag; it's the intended outcome of the revised aggregation.

## Step 4 — Convex hull classification (per unique location)

**Revised**: the hull is computed once per **unique location** (`dictionary_id` — 213 of them from the join, not per fuel-series, and not limited to the 189 with data), then `hull_status` is joined back onto every fuel-series at that location. This keeps hull membership a purely geometric property of the GB site layout — it shouldn't change depending on how many fuel-series a location happens to have, or whether that location has any pulled data at all.

Sites on the convex hull are always kept in training (a spatial model has no way to interpolate a boundary point from surrounding sites). Interior locations are eligible for holdout. Geometric classification only — no holdout proportions or thresholds are picked here.

In [10]:
from scipy.spatial import ConvexHull

unique_locations = site_bmu_map[["dictionary_id", "longitude", "latitude"]].drop_duplicates().reset_index(drop=True)
coords = unique_locations[["longitude", "latitude"]].to_numpy()
hull = ConvexHull(coords)
hull_idx = set(hull.vertices)

unique_locations["hull_status"] = ["boundary" if i in hull_idx else "interior" for i in range(len(unique_locations))]
print(f"Hull computed over {len(unique_locations)} unique locations (all locations from the join, with or without data)")
print(unique_locations["hull_status"].value_counts())

# Join hull_status back onto every fuel-series at each location
site_fuel_summary = site_fuel_summary.merge(
    unique_locations[["dictionary_id", "hull_status"]], on="dictionary_id", how="left"
)

n_boundary_locations = (unique_locations["hull_status"] == "boundary").sum()
n_boundary_locations_with_data = site_fuel_summary.loc[site_fuel_summary["hull_status"] == "boundary", "dictionary_id"].nunique()
print(f"\n{n_boundary_locations} boundary locations overall; {n_boundary_locations_with_data} of those have >= 1 fuel-series with data")
print(f"{len(site_fuel_summary)} fuel-series now carry a hull_status")

Hull computed over 213 unique locations (all locations from the join, with or without data)
hull_status
interior    202
boundary     11
Name: count, dtype: int64

11 boundary locations overall; 10 of those have >= 1 fuel-series with data
199 fuel-series now carry a hull_status


**Observations:** 11 boundary locations out of 213 total (unchanged count from before — makes sense, the hull's shape depends only on the set of unique coordinates, and that set didn't change). Of those 11 boundary locations, 10 have at least one fuel-series with data; the remaining 1 is one of the zero-data locations, so it's geometrically on the boundary but contributes nothing to training either way. 202 locations are interior (up from 178 previously, because this run counts *all* 213 locations, not just the 189 with data) — 189 of those interior locations actually have data and are holdout-eligible.

## Step 5 — Fuel-type coverage (revised granularity)

Coverage now counted per fuel-series, not per location — a multi-fuel location contributes one row per fuel type it actually has, instead of one blended `MIXED` row. Fuel types that were previously hidden inside a `MIXED` label (e.g. COAL, which only appeared combined with OCGT/BIOMASS before) can now show up in their own right. **Re-checked against the confirmed ≥7-interior-site stratification threshold** — WIND, CCGT, NPSHYD, NUCLEAR are already confirmed; this checks whether the revised granularity pushes any other fuel type over that line. No threshold is chosen here beyond that check — this remains a report for Simon to confirm the final holdout list against.

In [11]:
fuel_type_coverage = (
    site_fuel_summary.groupby(["fuel_type", "hull_status"]).size().unstack(fill_value=0)
)
for col in ["boundary", "interior"]:
    if col not in fuel_type_coverage.columns:
        fuel_type_coverage[col] = 0
fuel_type_coverage["total"] = fuel_type_coverage[["boundary", "interior"]].sum(axis=1)
fuel_type_coverage = fuel_type_coverage.sort_values("total", ascending=False)

fuel_type_coverage.to_csv(REF_DIR / "site_fuel_type_coverage.csv")

CONFIRMED_STRATIFIED = {"WIND", "CCGT", "NPSHYD", "NUCLEAR"}
INTERIOR_THRESHOLD = 7
newly_qualifying = fuel_type_coverage[
    (fuel_type_coverage["interior"] >= INTERIOR_THRESHOLD) & (~fuel_type_coverage.index.isin(CONFIRMED_STRATIFIED))
]

print(f"Confirmed stratified fuel types (>= {INTERIOR_THRESHOLD} interior sites): {sorted(CONFIRMED_STRATIFIED)}")
if len(newly_qualifying):
    print(f"\nFuel type(s) newly crossing the >= {INTERIOR_THRESHOLD} interior-site threshold under the revised granularity:")
    display(newly_qualifying)
else:
    print(f"\nNo additional fuel type crosses the >= {INTERIOR_THRESHOLD} interior-site threshold under the revised granularity.")

fuel_type_coverage

Confirmed stratified fuel types (>= 7 interior sites): ['CCGT', 'NPSHYD', 'NUCLEAR', 'WIND']

Fuel type(s) newly crossing the >= 7 interior-site threshold under the revised granularity:


hull_status,boundary,interior,total
fuel_type,,,
OCGT,1,11,12


hull_status,boundary,interior,total
fuel_type,,,
WIND,8,103,111
CCGT,0,38,38
NPSHYD,0,13,13
OCGT,1,11,12
NUCLEAR,1,7,8
COAL,0,5,5
BIOMASS,0,4,4
PS,0,4,4
RECIPROCATING,0,3,3


**Observations — OCGT newly qualifies for stratification:** splitting the `MIXED` locations back into per-fuel series pulled OCGT out from inside those blended labels and gave it **11 interior sites** (up from 3 previously, all inside `MIXED (CCGT/OCGT)`, `MIXED (COAL/OCGT)` etc.) — clearing the ≥7-interior-site threshold. **Simon: worth confirming whether OCGT should join WIND/CCGT/NPSHYD/NUCLEAR as a fifth stratified fuel type**, since the revised granularity is exactly why this changed.

COAL similarly re-emerged as its own category (5 interior sites) — still short of the threshold, but no longer invisible inside `MIXED` labels either.

Full revised table: WIND still dominates (111 series, 103 interior), then CCGT (38, all interior), NPSHYD (13), OCGT (12, 11 interior — see above), NUCLEAR (8, 7 interior), COAL (5), BIOMASS (4), PS (4), RECIPROCATING (3), and the single case-variant `Wind` row (1) — same data-quality note as before, still not corrected here.

## Step 6 — History-length distribution (revised granularity) and candidate cutoffs

Spread of available history per fuel-series, at the new granularity, plus a re-check of whether the ~2019-02-01 floor still holds. **No minimum-history cutoff is applied here** — instead, 2-3 candidate cutoffs are proposed below with the fuel-series (and whole locations) lost at each, for Simon to pick from.

In [12]:
site_fuel_summary["earliest_settlement_date"] = pd.to_datetime(site_fuel_summary["earliest_settlement_date"])
site_fuel_summary["latest_settlement_date"] = pd.to_datetime(site_fuel_summary["latest_settlement_date"])

history_stats = site_fuel_summary["earliest_settlement_date"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print("Per-fuel-series earliest available settlement date, distribution:")
print(history_stats)

print(f"\nEarliest start date across all fuel-series: {site_fuel_summary['earliest_settlement_date'].min().date()}")
print(f"Latest start date (shortest-history series, by start): {site_fuel_summary['earliest_settlement_date'].max().date()}")
print(f"Median start date: {site_fuel_summary['earliest_settlement_date'].median().date()}")

near_floor = site_fuel_summary["earliest_settlement_date"].between("2019-01-25", "2019-02-05")
print(f"\nFuel-series starting within +/-5 days of 2019-02-01: {near_floor.sum()} / {len(site_fuel_summary)} ({near_floor.mean():.1%})")

site_fuel_summary.groupby("fuel_type")["earliest_settlement_date"].agg(["min", "median", "max", "count"])

Per-fuel-series earliest available settlement date, distribution:
count                           199
mean     2019-03-10 17:29:14.773869
min             2019-02-01 00:00:00
10%             2019-02-01 00:00:00
25%             2019-02-01 00:00:00
50%             2019-02-01 00:00:00
75%             2019-02-01 00:00:00
90%             2019-02-01 00:00:00
max             2022-02-16 00:00:00
Name: earliest_settlement_date, dtype: object

Earliest start date across all fuel-series: 2019-02-01
Latest start date (shortest-history series, by start): 2022-02-16
Median start date: 2019-02-01

Fuel-series starting within +/-5 days of 2019-02-01: 189 / 199 (95.0%)


,min,median,max,count
fuel_type,,,,
BIOMASS,2019-02-01,2019-02-01,2019-02-01,4
CCGT,2019-02-01,2019-02-01,2019-02-01,38
COAL,2019-02-01,2019-02-01,2019-02-01,5
NPSHYD,2019-02-01,2019-02-01,2019-02-01,13
NUCLEAR,2019-02-01,2019-02-01,2019-02-01,8
OCGT,2019-02-01,2019-02-01,2019-02-01,12
PS,2019-02-01,2019-02-01,2019-02-01,4
RECIPROCATING,2019-02-01,2019-02-01,2019-02-01,3
WIND,2019-02-01,2019-02-01,2022-02-16,111


In [13]:
# Caveat check: EXTRACTION_START_YEAR (2015) bounds what we could possibly observe.
# If a meaningful number of series' earliest date sits right at that floor, it's a
# signal the true floor may be earlier still and the search window should be
# extended (the manifest makes that a cheap re-run, not a rebuild).
at_search_floor = site_fuel_summary["earliest_settlement_date"].between(f"{EXTRACTION_START_YEAR}-01-01", f"{EXTRACTION_START_YEAR}-01-07")
print(f"Fuel-series whose earliest date is within the first week of the {EXTRACTION_START_YEAR} search floor: {at_search_floor.sum()} / {len(site_fuel_summary)}")
if at_search_floor.sum() > 0:
    print("-> non-trivial: the true history floor for these series may predate our search window. Consider lowering EXTRACTION_START_YEAR and re-running (manifest will only fetch the newly-added earlier chunks).")
else:
    print("-> none/negligible: the search floor does not appear to be truncating real history.")

Fuel-series whose earliest date is within the first week of the 2015 search floor: 0 / 199
-> none/negligible: the search floor does not appear to be truncating real history.


### Candidate minimum-history cutoffs

History length per fuel-series is `latest_settlement_date - earliest_settlement_date` — using the span rather than just the start date, so a series that started on time but stopped early (e.g. a plant decommissioned mid-dataset) is correctly counted as short, not just late-starting ones. Three round-number candidates are proposed below (~1, ~2, ~3 years); **none is applied** — this is Simon's call.

In [14]:
site_fuel_summary["history_days"] = (
    site_fuel_summary["latest_settlement_date"] - site_fuel_summary["earliest_settlement_date"]
).dt.days

CANDIDATE_CUTOFFS_DAYS = [365, 730, 1095]  # ~1, ~2, ~3 years
total_series = len(site_fuel_summary)
total_locations = site_fuel_summary["dictionary_id"].nunique()

print(f"Baseline: {total_series} fuel-series across {total_locations} locations, no cutoff applied\n")
print("Candidate cutoffs (not applied - for Simon to choose):\n")
for cutoff in CANDIDATE_CUTOFFS_DAYS:
    below = site_fuel_summary["history_days"] < cutoff
    series_lost = below.sum()
    locations_after = site_fuel_summary.loc[~below, "dictionary_id"].nunique()
    locations_lost_entirely = total_locations - locations_after
    years = cutoff / 365
    print(
        f"- >= {cutoff} days (~{years:.0f} year{'s' if years != 1 else ''}): "
        f"{series_lost}/{total_series} fuel-series excluded, "
        f"{locations_lost_entirely}/{total_locations} locations would lose ALL their series"
    )

print("\nShortest 15 fuel-series by history length:")
site_fuel_summary[["site_fuel_id", "dictionary_id", "fuel_type", "history_days"]].sort_values("history_days").head(15)

Baseline: 199 fuel-series across 189 locations, no cutoff applied

Candidate cutoffs (not applied - for Simon to choose):

- >= 365 days (~1 year): 0/199 fuel-series excluded, 0/189 locations would lose ALL their series
- >= 730 days (~2 years): 0/199 fuel-series excluded, 0/189 locations would lose ALL their series
- >= 1095 days (~3 years): 3/199 fuel-series excluded, 2/189 locations would lose ALL their series

Shortest 15 fuel-series by history length:


,site_fuel_id,dictionary_id,fuel_type,history_days
6,10007_COAL,10007,COAL,972
7,10007_OCGT,10007,OCGT,972
79,10134_NUCLEAR,10134,NUCLEAR,1071
17,10021_CCGT,10021,CCGT,1330
109,10173_WIND,10173,WIND,1622
180,10291_WIND,10291,WIND,1622
196,10308_WIND,10308,WIND,1770
11,10012_COAL,10012,COAL,1771
12,10012_OCGT,10012,OCGT,1771
16,10014_OCGT,10014,OCGT,1794


**Observations:** the ~2019-02-01 floor holds at the new granularity too — 189/199 fuel-series (95.0%) start within 5 days of it, essentially unchanged from the location-level figure (94.7%). The search-floor caveat check again shows 0 series sitting at the 2015 search boundary, so `EXTRACTION_START_YEAR = 2015` still isn't truncating anything real.

**Candidate cutoffs are cheap here** — the data is unusually well-behaved: at ~1 year and ~2 years, *nothing* is lost (0/199 series, 0/189 locations). Only the ~3-year (1095-day) cutoff bites, and only lightly: 3 series excluded, and just 2 locations lose all their series entirely (Fiddlers Ferry-style plants that stopped generating partway through the dataset, plus one short-history nuclear/CCGT series). Given how little is at stake, the main judgement call for Simon isn't really "how much do we lose" so much as "do we want the safety margin of a longer minimum" for model training stability.

## Step 7 — Parquet consolidation (derived cache, re-runnable)

Reads every per-fuel-series CSV under `data/interim/site_generation/` (now named `{dictionary_id}_{fuel_type}.csv`) and writes one consolidated parquet file for the training notebook to load quickly. **The CSVs remain the source of truth** — this file is a disposable, regenerable convenience cache. Re-running this cell after further extraction progress just picks up whatever series CSVs exist at that point.

In [15]:
def consolidate_to_parquet(out_path=INTERIM_DIR / "site_generation_consolidated.parquet"):
    site_files = sorted(SITE_GEN_DIR.glob("*.csv"))
    if not site_files:
        print("No fuel-series CSVs found yet - nothing to consolidate.")
        return None

    frames = [pd.read_csv(f) for f in site_files]
    consolidated = pd.concat(frames, ignore_index=True)

    # Attach site_fuel_id, hull_status and name for convenience at load time in the training notebook
    consolidated = consolidated.merge(
        site_fuel_summary[["dictionary_id", "fuel_type", "site_fuel_id", "hull_status", "name"]],
        on=["dictionary_id", "fuel_type"], how="left",
    )
    consolidated.to_parquet(out_path, index=False)
    print(
        f"Wrote {len(consolidated):,} rows across {consolidated['site_fuel_id'].nunique()} fuel-series "
        f"({consolidated['dictionary_id'].nunique()} locations) to {out_path}"
    )
    print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")
    return consolidated


consolidated = consolidate_to_parquet()

Wrote 25,069,098 rows across 199 fuel-series (189 locations) to ../data/interim/site_generation_consolidated.parquet
File size: 66.4 MB


**Observations:** ~25.1M rows across 199 fuel-series (189 locations) in a 66.4MB parquet file — barely larger than the previous location-level version (23.98M rows, 65.5MB) despite splitting the 13 multi-fuel locations into separate series, since it's the same underlying half-hourly data either way, just labelled differently. Still comfortably small enough to load in full.

## Stopping point

Confirmed and applied this revision:

1. **Aggregation unit**: `(dictionary_id, fuel_type)`, not `dictionary_id` alone — individual fuel signals preserved, not blended. See Step 3.
2. **Split key**: `dictionary_id` remains it — every fuel-series at one location moves to train/test as one atomic block.
3. **Convex hull**: computed once per unique location (213), joined back by `dictionary_id`. See Step 4.
4. **Stratified fuel types**: re-checked against the ≥7-interior-site threshold at the new granularity. See Step 5.

Still pending Simon's confirmation, not decided here:

1. **Minimum-history cutoff** — Step 6 proposes three candidate cutoffs with the fuel-series and locations lost at each; none applied.
2. **Parquet as anything other than a derived cache** — still just a regenerable convenience file, safe to delete and rebuild.

Also still true from the original brief: the mixed-fleet-scope tension (a single-fuel-type dataset would tell a cleaner geographic story) needs to be stated plainly in the eventual write-up, not just here.